# Flight Delays · cost-sensitive retraining (Spark, 5.8 млн рейсов)

**Данные и задача не меняются:** `usdot/flight-delays` (BTS 2015); прогнозируем прибытие с задержкой ≥15 минут.
Цены ошибок остаются прежними: **FP = $25, FN = $100**. Порог оптимизируется по денежной стоимости, а не фиксируется на 0.5.

## Что изменено в обучении

1. Добавлены признаки взаимодействия `перевозчик × маршрут`, `перевозчик × месяц/час`, `аэропорт × час`. Target encoding рассчитывается **только по train** со сглаживанием и leave-one-out для строк train — метки validation/test в признаки не попадают.
2. Время кодируется циклически (час, день недели, месяц), добавлены пиковые интервалы, логарифмы плановой загрузки аэропортов (из расписания, без фактических статусов рейса) и нормированная скорость по расписанию.
3. Вместо единственного набора параметров проверяются исходный GBT, два регуляризованных GBT, GBT с cost-sensitive весами 4:1, RandomForest и LogReg.
4. Выбор конфигурации и порога выполняется по **validation cost/1000**. Test не участвует в выборе — он нужен только для финальной оценки выбранных конфигураций. Разбиение выполняется целыми днями, но не является хронологическим holdout.
5. Структура `report.json` и ключевые поля метрик сохранены для текущего фронтенда. Дополнительные поля и `training_experiments.csv` — только для аудита эксперимента.

**Важно:** этот файл содержит код нового перерасчёта, а не заранее посчитанные новые результаты. Для получения новых цифр запустите ноутбук на том же полном Spark-кластере и том же исходном CSV. Числа старого прогона нельзя выдавать за результат новой модели.

**Ресурсы:** Spark 3.5.x; ожидается профиль `spark` из проекта. При `SPARK_MODE=local` данные сокращаются, поэтому такие цифры нельзя сравнивать с полным прогоном.

In [ ]:
import os, re, sys, json, math, time, socket, shutil, warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 110


def find_project_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for parent in [p, *p.parents]:
        if (parent / 'requirements.txt').exists() and (parent / 'notebooks').exists():
            return parent
    raise RuntimeError('Корень проекта не найден')


SEED    = 42
PROJECT = find_project_root()
TASK    = 'flight_delays'

MODEL_DIR = PROJECT / 'models' / 'classification' / TASK
ART_DIR   = PROJECT / 'artifacts' / 'classification' / TASK
MODEL_DIR.mkdir(parents=True, exist_ok=True)
ART_DIR.mkdir(parents=True, exist_ok=True)

print('PROJECT  :', PROJECT)
print('MODEL_DIR:', MODEL_DIR)
print('ART_DIR  :', ART_DIR)

## 1. Метрики без Spark (numpy)

Spark считает **одну гистограмму** скоров (1001 бин × 2 класса) на валидации и тесте. Из неё уже на драйвере, мгновенно и точно до 0.001,
получаем confusion matrix при любом пороге, кривые ROC/PR, лучший порог и все метрики. Это заменяет сотни повторных проходов по данным.
Функции проверены против `sklearn` (расхождение ~1e-16).

Дополнительно к той же гистограмме Spark считает **суммы score и score²** по бинам — из них получаем **точный Brier Score**
и диаграмму калибровки без повторных проходов. **AP** считается по гистограмме ступенчатой формулой `Σ (Rₙ − Rₙ₋₁)·Pₙ`
(как `sklearn.average_precision_score`, разрешение порога 0.001).

In [ ]:
# ---------- метрики без Spark: считаются по гистограммам / confusion matrix ----------
import math, json
import numpy as np

NBINS = 1000          # разрешение порога = 0.001


def _div(a, b):
    return a / b if b else 0.0


def _fbeta(p, r, beta):
    d = beta * beta * p + r
    return (1 + beta * beta) * p * r / d if d else 0.0


def binary_metrics(tp, fp, fn, tn):
    """Все метрики бинарной классификации из четырёх чисел confusion matrix."""
    tp, fp, fn, tn = (float(x) for x in (tp, fp, fn, tn))
    n, P, N = tp + fp + fn + tn, tp + fn, tn + fp
    prec_p, rec_p = _div(tp, tp + fp), _div(tp, P)
    prec_n, rec_n = _div(tn, tn + fn), _div(tn, N)
    f1p, f05p, f2p = (_fbeta(prec_p, rec_p, b) for b in (1.0, 0.5, 2.0))
    f1n, f05n, f2n = (_fbeta(prec_n, rec_n, b) for b in (1.0, 0.5, 2.0))
    acc = _div(tp + tn, n)
    den = math.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    mcc = _div(tp * tn - fp * fn, den)
    pe = _div((tp + fp) * P + (tn + fn) * N, n * n)
    kappa = _div(acc - pe, 1 - pe)
    return {
        'accuracy': acc,
        'balanced_accuracy': (rec_p + rec_n) / 2,
        'precision_macro': (prec_p + prec_n) / 2,
        'recall_macro': (rec_p + rec_n) / 2,
        'f1_macro': (f1p + f1n) / 2,
        'f1_weighted': _div(f1p * P + f1n * N, n),
        'f0.5_macro': (f05p + f05n) / 2,
        'f2_macro': (f2p + f2n) / 2,
        'mcc': mcc,
        'kappa': kappa,
        'specificity': rec_n,
        'sensitivity': rec_p,
        'precision_pos': prec_p,
        'recall_pos': rec_p,
        'f1_pos': f1p,
        'f0.5_pos': f05p,
        'f2_pos': f2p,
        # отрицательный класс («вовремя») как целевой: precision = NPV, recall = specificity
        'npv': prec_n,
        'precision_neg': prec_n,
        'recall_neg': rec_n,
        'f1_neg': f1n,
        'f0.5_neg': f05n,
        'f2_neg': f2n,
        'fpr': _div(fp, N),
    }


def hist_counts(neg, pos):
    """neg/pos — гистограммы длины NBINS+1 (бин = floor(score*NBINS)).
    Для порога k/NBINS (k = 0..NBINS) объект предсказан положительным при score >= k/NBINS.
    Возвращает массивы tp, fp, fn, tn длины NBINS+1."""
    neg = np.asarray(neg, dtype=np.float64)
    pos = np.asarray(pos, dtype=np.float64)
    neg_ge = np.cumsum(neg[::-1])[::-1]
    pos_ge = np.cumsum(pos[::-1])[::-1]
    P, N = pos.sum(), neg.sum()
    return pos_ge, neg_ge, P - pos_ge, N - neg_ge          # tp, fp, fn, tn


def _mcc_vec(tp, fp, fn, tn):
    num = tp * tn - fp * fn
    den = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    return np.divide(num, den, out=np.zeros_like(num, dtype=np.float64), where=den > 0)


F_KEYS = ('f0.5_pos', 'f2_pos', 'f0.5_neg', 'f2_neg')


def _f_vec(p, r, beta):
    b2 = beta * beta
    den = b2 * p + r
    return np.divide((1 + b2) * p * r, den, out=np.zeros_like(den, dtype=np.float64), where=den > 0)


def f_curves(neg, pos):
    """F0.5 и F2 для всех порогов k/NBINS — для положительного и отрицательного класса.
    Положительный: precision = TP/(TP+FP), recall = TP/P. Отрицательный: precision = NPV = TN/(TN+FN), recall = TN/N."""
    tp, fp, fn, tn = hist_counts(neg, pos)
    z = lambda a, b: np.divide(a, b, out=np.zeros_like(a, dtype=np.float64), where=b > 0)
    prec_p, rec_p = z(tp, tp + fp), z(tp, tp + fn)
    prec_n, rec_n = z(tn, tn + fn), z(tn, tn + fp)
    return {'f0.5_pos': _f_vec(prec_p, rec_p, 0.5), 'f2_pos': _f_vec(prec_p, rec_p, 2.0),
            'f0.5_neg': _f_vec(prec_n, rec_n, 0.5), 'f2_neg': _f_vec(prec_n, rec_n, 2.0)}


def best_threshold_k(neg, pos, objective='mcc', cost_fp=1.0, cost_fn=1.0):
    """Индекс k лучшего порога на (валидационной) гистограмме.
    objective: 'mcc' (максимум) | 'cost' (минимум fp*cost_fp + fn*cost_fn) |
               'f0.5_pos' | 'f2_pos' | 'f0.5_neg' | 'f2_neg' (максимум F-меры нужного класса)."""
    tp, fp, fn, tn = hist_counts(neg, pos)
    if objective == 'mcc':
        return int(np.argmax(_mcc_vec(tp, fp, fn, tn)))
    if objective == 'cost':
        return int(np.argmin(fp * cost_fp + fn * cost_fn))
    if objective in F_KEYS:
        return int(np.argmax(f_curves(neg, pos)[objective]))
    raise ValueError(objective)


def val_best_block(neg_v, pos_v):
    """Лучшие пороги по F0.5/F2 (оба класса), подобранные на ВАЛИДАЦИИ."""
    fc = f_curves(neg_v, pos_v)
    out = {}
    for key in F_KEYS:
        k = int(np.argmax(fc[key]))
        out[key] = {'threshold': k / NBINS, 'val_score': float(fc[key][k])}
    return out


def average_precision(neg, pos):
    """Average Precision = Σ (R_k − R_{k+1}) · P_k по порогам k/NBINS от мягкого к жёсткому (как sklearn)."""
    tp, fp, fn, tn = hist_counts(neg, pos)
    P = tp[0]
    if P <= 0:
        return 0.0
    prec = np.divide(tp, tp + fp, out=np.zeros_like(tp, dtype=np.float64), where=(tp + fp) > 0)
    rec = tp / P
    rec_next = np.append(rec[1:], 0.0)                 # recall при более строгом пороге; за последним — 0
    return float(np.sum((rec - rec_next) * prec))


def brier_from_moments(neg, pos, mom):
    """Точный Brier Score = mean((score − y)²) из сумм по бинам: Σ(s²) − 2·Σ_{y=1}(s) + n_pos."""
    n = float(neg.sum() + pos.sum())
    num = mom['s2_neg'].sum() + mom['s2_pos'].sum() - 2.0 * mom['s1_pos'].sum() + pos.sum()
    return float(num / n) if n else float('nan')


def calibration_block(neg, pos, mom, n_bins=10):
    """Диаграмма калибровки: n_bins равных интервалов score → (средний score, доля задержек, размер)."""
    g = np.minimum((np.arange(NBINS + 1) * n_bins) // NBINS, n_bins - 1)
    n = np.bincount(g, weights=neg + pos, minlength=n_bins)
    n_pos = np.bincount(g, weights=pos, minlength=n_bins)
    s1 = np.bincount(g, weights=mom['s1_neg'] + mom['s1_pos'], minlength=n_bins)
    ok = n > 0
    mean_pred, obs = s1[ok] / n[ok], n_pos[ok] / n[ok]
    ece = float(np.sum(n[ok] / n.sum() * np.abs(obs - mean_pred)))
    return {'mean_pred': mean_pred, 'obs_rate': obs, 'n': n[ok], 'ece': ece}


def sweep_block(neg, pos, n_points=101):
    """Подвыборка порогов 0..1 для фронта: счётчики tp/fp/fn/tn на тестовой гистограмме."""
    tp, fp, fn, tn = hist_counts(neg, pos)
    ks = np.unique(np.round(np.linspace(0, NBINS, n_points)).astype(int))
    return {
        'thresholds': [round(float(k) / NBINS, 4) for k in ks],
        'tp': [int(tp[k]) for k in ks], 'fp': [int(fp[k]) for k in ks],
        'fn': [int(fn[k]) for k in ks], 'tn': [int(tn[k]) for k in ks],
    }


def _thin(x, y):
    out_x, out_y = [], []
    for a, b in zip(x, y):
        if out_x and abs(a - out_x[-1]) < 2e-3 and abs(b - out_y[-1]) < 2e-3:
            continue
        out_x.append(round(float(a), 4)); out_y.append(round(float(b), 4))
    return out_x, out_y


def _cap(x, y, n=140):
    if len(x) <= n:
        return x, y
    idx = np.unique(np.linspace(0, len(x) - 1, n).round().astype(int))
    return [x[i] for i in idx], [y[i] for i in idx]


def curves_block(neg, pos):
    """ROC и PR-кривые по гистограмме (для графиков; точные AUC считает Spark)."""
    tp, fp, fn, tn = hist_counts(neg, pos)
    P, N = max(tp[0], 1.0), max(fp[0], 1.0)
    tpr, fpr = tp / P, fp / N
    prec = np.divide(tp, tp + fp, out=np.ones_like(tp, dtype=np.float64), where=(tp + fp) > 0)
    fx, fy = _cap(*_thin(fpr[::-1], tpr[::-1]))
    rx, ry = _cap(*_thin(tpr[::-1], prec[::-1]))
    return {'roc': {'fpr': fx, 'tpr': fy}, 'pr': {'recall': rx, 'precision': ry}}


def multiclass_metrics(cm):
    """cm[i][j] — сколько объектов истинного класса i предсказано как j."""
    cm = np.asarray(cm, dtype=np.float64)
    K, s = cm.shape[0], cm.sum()
    t, p, d = cm.sum(axis=1), cm.sum(axis=0), np.diag(cm)
    prec = np.divide(d, p, out=np.zeros(K), where=p > 0)
    rec = np.divide(d, t, out=np.zeros(K), where=t > 0)
    f1 = np.divide(2 * prec * rec, prec + rec, out=np.zeros(K), where=(prec + rec) > 0)
    acc = d.sum() / s
    mcc_den = math.sqrt(max((s * s - (p * p).sum()) * (s * s - (t * t).sum()), 0.0))
    mcc = (d.sum() * s - (p * t).sum()) / mcc_den if mcc_den else 0.0
    pe = (p * t).sum() / (s * s)
    kappa = (acc - pe) / (1 - pe) if pe < 1 else 0.0
    idx = np.arange(K)
    w = (idx[:, None] - idx[None, :]) ** 2 / max((K - 1) ** 2, 1)
    expected = np.outer(t, p) / s
    qwk = 1 - (w * cm).sum() / (w * expected).sum() if (w * expected).sum() else 0.0
    return {
        'accuracy': acc, 'balanced_accuracy': rec.mean(),
        'precision_macro': prec.mean(), 'recall_macro': rec.mean(), 'f1_macro': f1.mean(),
        'precision_weighted': (prec * t).sum() / s, 'recall_weighted': (rec * t).sum() / s,
        'f1_weighted': (f1 * t).sum() / s, 'mcc': mcc, 'kappa': kappa, 'qwk': qwk,
        '_per_class': {'precision': prec, 'recall': rec, 'f1': f1, 'support': t},
    }


def to_native(o):
    """Рекурсивно приводит numpy/NaN/Inf к JSON-совместимым значениям (NaN → null)."""
    if isinstance(o, dict):
        return {str(k): to_native(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [to_native(v) for v in o]
    if isinstance(o, np.ndarray):
        return to_native(o.tolist())
    if isinstance(o, np.generic):
        return to_native(o.item())
    if isinstance(o, float):
        return None if (math.isnan(o) or math.isinf(o)) else o
    if hasattr(o, '__fspath__'):
        return str(o)
    return o


def dump_json(obj, path):
    path.write_text(json.dumps(to_native(obj), indent=2, ensure_ascii=False, allow_nan=False), encoding='utf-8')

## 2. Spark: сессия, ресурсы, общий каталог

Используется тот же Spark-кластер и общий каталог проекта. Полный прогон (`DATA_FRACTION=1.0`) обязателен для сопоставления с прежними 5.71 млн очищенных строк.

In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.ml.functions import vector_to_array
from pyspark import StorageLevel
import requests

# ---- бюджет ресурсов берём из docker-compose ----
# spark-worker:  command --cores 8 --memory 10g  (limit cpus=8, memory=12g)
# ml (driver):   лимит cpus=3.0, memory=6g       → драйвер 2g, локально не считаем
SPARK_MASTER_URL = os.environ.get('SPARK_MASTER_URL', 'spark://spark-master:7077')
CORES      = int(os.environ.get('SPARK_CORES', os.environ.get('WEATHER_SPARK_CORES', '8')))                 # worker: --cores 8
EXEC_MEM   = os.environ.get('SPARK_EXECUTOR_MEMORY', os.environ.get('WEATHER_SPARK_EXECUTOR_MEMORY', '8g'))  # worker: --memory 10g, limit 12g
DRIVER_MEM = os.environ.get('SPARK_DRIVER_MEMORY', '2g')       # контейнер ml ограничен 6g — драйвер держим скромным
LOCAL_CORES = int(os.environ.get('SPARK_LOCAL_CORES', '3'))    # контейнер ml: cpus 3.0 (только для SPARK_MODE=local)
FORCE_MODE = os.environ.get('SPARK_MODE', 'cluster')           # cluster (по умолчанию) | auto | local
ALLOW_LOCAL_FALLBACK = os.environ.get('ALLOW_LOCAL_FALLBACK', '0') == '1'
EVENT_DIR  = Path('/opt/spark/events')                         # общий каталог для Spark History Server


def _master_tcp_ok(url, timeout=2.0):
    try:
        host, port = url.split('://', 1)[1].split(':')
        with socket.create_connection((host, int(port)), timeout=timeout):
            return True
    except Exception:
        return False


def _alive_workers(url):
    """Список живых worker из Web UI мастера (8085). None — UI недоступен."""
    try:
        host = url.split('://', 1)[1].split(':')[0]
        info = requests.get(f'http://{host}:8085/json/', timeout=3).json()
        return [w for w in info.get('workers', []) if w.get('state') == 'ALIVE']
    except Exception:
        return None


def choose_mode():
    """
    cluster (default) — требуем живой spark-master с ALIVE-worker. Нет → RuntimeError.
    auto             — пробуем cluster, при неудаче fallback на local (с громким предупреждением).
    local            — принудительно local[3] (для отладки; урезает данные, ест 3 ядра ml).
    ALLOW_LOCAL_FALLBACK=1 дополнительно разрешает fallback в режиме cluster.
    """
    if FORCE_MODE == 'local':
        print(f'⚠ SPARK_MODE=local: работаем на {LOCAL_CORES} ядрах ml (данные урежутся, только для отладки)')
        return 'local'

    if FORCE_MODE not in ('cluster', 'auto'):
        raise ValueError(f'SPARK_MODE={FORCE_MODE!r} (ожидалось auto|cluster|local)')

    if _master_tcp_ok(SPARK_MASTER_URL):
        workers = _alive_workers(SPARK_MASTER_URL)
        if workers is None or len(workers) > 0:
            return 'cluster'
        print('⚠ spark-master отвечает, но ALIVE-worker нет — задачи выполнять некому')

    if FORCE_MODE == 'auto' or ALLOW_LOCAL_FALLBACK:
        print(f'⚠ падаю в fallback local[{LOCAL_CORES}] — это НЕ полный прогон, 3 ядра ml, DATA_FRACTION=0.15')
        return 'local'

    raise RuntimeError(
        'Spark-кластер недоступен, а SPARK_MODE=cluster (по умолчанию).\n'
        f'  master: {SPARK_MASTER_URL}\n'
        '  Запустите профиль:\n'
        '      docker compose --profile spark up -d\n'
        '  Проверить: http://localhost:8085 (master UI), http://localhost:8086 (worker UI)\n'
        '  Для локальной отладки без кластера:\n'
        '      SPARK_MODE=local     (урежет данные до 15%, 3 ядра ml)\n'
        '      SPARK_MODE=auto      (попробует cluster, потом local)\n'
    )


def start_spark(mode):
    active = SparkSession.getActiveSession()
    if active is not None:
        active.stop()
    b = (SparkSession.builder.appName(f'ml-playground · {TASK}')
         .config('spark.ui.showConsoleProgress', 'false')
         .config('spark.sql.adaptive.enabled', 'true')
         .config('spark.sql.session.timeZone', 'UTC')
         .config('spark.driver.maxResultSize', '512m'))
    if mode == 'cluster':
        b = (b.master(SPARK_MASTER_URL)
              .config('spark.submit.deployMode', 'client')     # driver — здесь, executor — на worker
              .config('spark.driver.memory', DRIVER_MEM)
              .config('spark.driver.host', socket.gethostbyname(socket.gethostname()))
              .config('spark.driver.bindAddress', '0.0.0.0')
              .config('spark.driver.port', '7078')
              .config('spark.blockManager.port', '7079')
              .config('spark.executor.memory', EXEC_MEM)       # 8g
              .config('spark.executor.cores', str(CORES))      # 8
              .config('spark.cores.max', str(CORES))           # 8 → 1 executor × 8 cores на worker
              .config('spark.sql.shuffle.partitions', str(CORES * 6))   # 48
              .config('spark.default.parallelism', str(CORES * 6)))     # 48
    else:
        b = (b.master(f'local[{LOCAL_CORES}]')
              .config('spark.driver.memory', '4g')
              .config('spark.sql.shuffle.partitions', str(LOCAL_CORES * 4)))
    if EVENT_DIR.is_dir():
        b = b.config('spark.eventLog.enabled', 'true').config('spark.eventLog.dir', f'file://{EVENT_DIR}')
    spark = b.getOrCreate()
    spark.sparkContext.setLogLevel('WARN')
    return spark


# ---------- ОБЩИЙ КАТАЛОГ (виден драйверу И executor-ам) ----------
# По docker-compose.yml в оба контейнера (ml и spark-worker) смонтированы:
#   ./data               → /opt/spark/data      (ml + worker)
#   ./models             → /app/models          (ml + worker)
#   ./spark-state/events → /opt/spark/events    (ml + worker)
# Плюс в ml есть `.:/app`, поэтому /app/data в ml — это ТОТ ЖЕ хост-каталог ./data,
# что и /opt/spark/data в ml и worker. Именно поэтому копировать ./data/flight-delays
# в ./data/spark_cache/raw/ незачем: worker уже видит его по /opt/spark/data/flight-delays.
#
# Имя проверочного файла — probe.txt, БЕЗ префикса '_'.
# Hadoop FileInputFormat молча игнорирует файлы с '_' или '.' в начале имени
# (то же правило прячет _SUCCESS и .crc в parquet), поэтому
#   spark.read.text('file:///.../_probe.txt').collect() == []
# и раньше это давало ложное «executor не видят каталог».
SHARED_CANDIDATES = [
    Path('/opt/spark/data/spark_cache'),       # ./data               → ml + worker
    Path('/app/models/.spark_cache'),          # ./models             → ml + worker
    Path('/opt/spark/events/spark_cache'),     # ./spark-state/events → ml + worker
    PROJECT / 'data' / 'spark_cache',          # fallback для local[3] / нестандартного compose
]


def pick_shared_dir(spark, mode):
    if mode == 'local':
        d = PROJECT / 'data' / 'spark_cache'
        d.mkdir(parents=True, exist_ok=True)
        print(f'✓ shared dir: {d} — local[3], executor в том же процессе')
        return d
    for d in SHARED_CANDIDATES:
        try:
            d.mkdir(parents=True, exist_ok=True)
            probe = d / 'probe.txt'                # НЕ _probe.txt (см. комментарий выше)
            probe.write_text('ok')
            got = spark.read.text(f'file://{probe}').collect()   # читает executor, не драйвер
            if got and got[0][0] == 'ok':
                print(f'✓ shared dir: {d} — виден executor-ам')
                return d
            print(f'  {d}: probe прочитан пустым (got={got!r})')
        except Exception as e:
            print(f'  {d}: executor не видят каталог ({type(e).__name__})')
    return None


MODE  = choose_mode()
spark = start_spark(MODE)
SHARED = pick_shared_dir(spark, MODE)

# fallback в local разрешён только через choose_mode (по SPARK_MODE/ALLOW_LOCAL_FALLBACK)
# если сюда попали с MODE=cluster и SHARED=None — это проблема монтирования, а не режима
if SHARED is None:
    raise RuntimeError(
        'Ни один общий каталог не виден executor-ам. Проверьте docker-compose:\n'
        '  spark-worker должен монтировать ./data, ./models или ./spark-state/events\n'
        '  по тому же пути, что и ml (см. SHARED_CANDIDATES).\n'
        '  Кандидаты: ' + ', '.join(str(p) for p in SHARED_CANDIDATES)
    )

sc = spark.sparkContext
DATA_FRACTION = float(os.environ.get('DATA_FRACTION', '1.0' if MODE == 'cluster' else '0.15'))

SPARK_INFO = {
    'mode': MODE,
    'master': SPARK_MASTER_URL if MODE == 'cluster' else f'local[{LOCAL_CORES}]',
    'spark_version': spark.version,
    'app_id': sc.applicationId,
    'cores': CORES if MODE == 'cluster' else LOCAL_CORES,
    'executor_memory': EXEC_MEM if MODE == 'cluster' else '4g (driver)',
    'default_parallelism': int(sc.defaultParallelism),
    'shuffle_partitions': int(spark.conf.get('spark.sql.shuffle.partitions')),
    'shared_dir': str(SHARED),
    'data_fraction': DATA_FRACTION,
}
print(json.dumps(SPARK_INFO, indent=2, ensure_ascii=False))
if MODE == 'local':
    print(f'\n⚠ local-режим: данные урезаны до {DATA_FRACTION:.0%} (DATA_FRACTION), используются {LOCAL_CORES} ядра ml. '
          'Для полного объёма на 8 ядрах spark-worker: docker compose --profile spark up -d и SPARK_MODE=cluster.')


# ---------- утилиты ----------
TIMINGS = {}


class stage:
    """with stage('имя'): ...  — печатает и копит время этапов (идёт в report.json)."""
    def __init__(self, name):
        self.name = name

    def __enter__(self):
        self.t = time.time()
        print(f'▶ {self.name} ...')
        return self

    def __exit__(self, *exc):
        TIMINGS[self.name] = round(time.time() - self.t, 1)
        print(f'✔ {self.name}: {TIMINGS[self.name]} с')


def ensure_shared_dir(src):
    """
    Приводит src к пути, видимому executor-ам (под SHARED / /opt/spark/data).
    Копирует только когда действительно надо.

    Ключевой нюанс: в ml есть ДВА монтирования на один и тот же хост-каталог ./data —
       .:/app               → /app/data       (= ./data на хосте)
       ./data:/opt/spark/data → /opt/spark/data (= ./data на хосте)
    Поэтому /app/data/flight-delays и /opt/spark/data/flight-delays — ОДИН И ТОТ ЖЕ файл
    на хосте. spark-worker монтирует только ./data:/opt/spark/data, значит видит его как
    /opt/spark/data/flight-delays. Копирование в SHARED/raw/ тут не нужно и только плодит
    дубли; а попытка copytree на несуществующем пути — это та самая FileNotFoundError.
    """
    src = Path(src)
    try:
        src = src.resolve()
    except OSError:
        pass
    shared = SHARED.resolve()

    # 1) Уже под SHARED — отлично, ничего не делаем
    try:
        src.relative_to(shared)
        if src.exists():
            return src
    except ValueError:
        pass

    # 2) /app/data/... и /opt/spark/data/... — тот же хост-каталог ./data, что и у worker-а.
    #    Пересчитываем на /opt/spark/data/<rel> без копирования (работает и для файлов, и для каталогов).
    for alias in (Path('/app/data'), Path('/opt/spark/data')):
        try:
            rel = src.relative_to(alias.resolve())
        except ValueError:
            continue
        worker_path = Path('/opt/spark/data') / rel
        if worker_path.exists():
            print(f'✓ {src} → {worker_path} (тот же хост-каталог ./data, копия не нужна)')
            return worker_path

    # 3) Если src вообще не существует — внятная ошибка вместо падения copytree
    if not src.exists():
        raise FileNotFoundError(
            f'Нет данных: {src}\n'
            f'  kagglehub недоступен и локальной копии нет.\n'
            f'  Положите файлы на хост, например:\n'
            f'    ./data/{src.name}/    (в контейнере это /opt/spark/data/{src.name}/)\n'
            f'  или настройте доступ к Kaggle (KAGGLE_USERNAME / KAGGLE_KEY).\n'
        )

    # 4) Обычный кейс — копируем под SHARED/raw/<name>
    dst = shared / 'raw' / src.name
    if src.is_file():
        dst.parent.mkdir(parents=True, exist_ok=True)
        if dst.exists() and dst.resolve() == src:
            return src
        print(f'копирую файл {src} → {dst} (чтобы executor увидели файл)')
        shutil.copy2(src, dst)
        return dst
    if dst.exists() and dst.resolve() == src:
        return src
    print(f'копирую {src} → {dst} (чтобы executor увидели файлы)')
    shutil.copytree(src, dst, dirs_exist_ok=True)
    return dst


def spark_path(p):
    return f'file://{p}'

In [ ]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator


def score_sdf(model, sdf, extra=()):
    """Предсказания модели → DataFrame(label, *extra, score = P(класс 1))."""
    return model.transform(sdf).select('label', *extra, vector_to_array('probability')[1].alias('score'))


def score_histogram(sdf, moments=False):
    """Гистограмма score по NBINS+1 бинам отдельно для классов 0 и 1 — один проход по данным.
    Из неё без повторных заданий получаем confusion matrix при ЛЮБОМ пороге k/NBINS.
    moments=True дополнительно возвращает суммы score и score² по бинам (для точного Brier Score и калибровки)."""
    b = F.greatest(F.lit(0), F.least(F.floor(F.col('score') * NBINS).cast('int'), F.lit(NBINS)))
    g = (sdf.select(b.alias('b'), F.col('label').cast('int').alias('y'), F.col('score').alias('s'))
            .groupBy('b', 'y')
            .agg(F.count('*').alias('c'), F.sum('s').alias('s1'), F.sum(F.col('s') * F.col('s')).alias('s2')))
    rows = g.collect()
    neg, pos = np.zeros(NBINS + 1), np.zeros(NBINS + 1)
    mom = {k: np.zeros(NBINS + 1) for k in ('s1_neg', 's1_pos', 's2_neg', 's2_pos')}
    for r in rows:
        side = 'pos' if r['y'] == 1 else 'neg'
        (pos if side == 'pos' else neg)[r['b']] = r['c']
        mom[f's1_{side}'][r['b']] = r['s1']
        mom[f's2_{side}'][r['b']] = r['s2']
    return (neg, pos, mom) if moments else (neg, pos)


def spark_auc(sdf):
    """Точные ROC-AUC и PR-AUC (без даунсэмплинга кривых) средствами Spark."""
    out = {}
    for metric, key in (('areaUnderROC', 'roc_auc'), ('areaUnderPR', 'pr_auc')):
        try:
            ev = BinaryClassificationEvaluator(rawPredictionCol='score', labelCol='label', metricName=metric, numBins=0)
        except TypeError:
            ev = BinaryClassificationEvaluator(rawPredictionCol='score', labelCol='label', metricName=metric)
        out[key] = float(ev.evaluate(sdf))
    return out


def tree_importance(model, names):
    imp = model.featureImportances.toArray()
    return sorted([{'name': n, 'value': float(v)} for n, v in zip(names, imp)], key=lambda d: -d['value'])

## 3. Загрузка и очистка данных

Сохраняем прежнее определение целевого класса и правила исключения отменённых/перенаправленных рейсов. Разбиение по календарным дням остаётся детерминированным.

In [ ]:
os.environ['KAGGLEHUB_CACHE'] = str(SHARED / 'kagglehub')
import kagglehub

with stage('Загрузка usdot/flight-delays'):
    try:
        DATA_DIR = Path(kagglehub.dataset_download('usdot/flight-delays'))
    except Exception as e:
        print('kagglehub недоступен:', e)
        DATA_DIR = PROJECT / 'data' / 'flight-delays'
        print('использую локальную копию:', DATA_DIR)

DATA_DIR = ensure_shared_dir(DATA_DIR)
flights_csv, airlines_csv = DATA_DIR / 'flights.csv', DATA_DIR / 'airlines.csv'
assert flights_csv.exists(), f'нет файла {flights_csv}'
print(f'flights.csv: {flights_csv.stat().st_size / 1e6:.0f} МБ')

raw = spark.read.option('header', True).csv(spark_path(flights_csv))
print('столбцов:', len(raw.columns))

In [ ]:
sel = raw.select(
    F.col('MONTH').cast('int').alias('month'),
    F.col('DAY').cast('int').alias('day'),
    F.col('DAY_OF_WEEK').cast('int').alias('dow'),
    F.col('AIRLINE').alias('airline'),
    F.col('ORIGIN_AIRPORT').alias('origin'),
    F.col('DESTINATION_AIRPORT').alias('dest'),
    F.col('SCHEDULED_DEPARTURE').cast('int').alias('sched_dep'),
    F.col('SCHEDULED_ARRIVAL').cast('int').alias('sched_arr'),
    F.col('SCHEDULED_TIME').cast('double').alias('sched_time'),
    F.col('DISTANCE').cast('double').alias('distance'),
    F.col('ARRIVAL_DELAY').cast('double').alias('arr_delay'),
    F.col('CANCELLED').cast('int').alias('cancelled'),
    F.col('DIVERTED').cast('int').alias('diverted'),
)

with stage('Подсчёт строк в исходном CSV'):
    N_RAW = sel.count()
print(f'строк в CSV: {N_RAW:,}')

flown = (sel
    .where((F.col('cancelled') == 0) & (F.col('diverted') == 0)
           & F.col('arr_delay').isNotNull() & F.col('sched_time').isNotNull()
           & F.col('distance').isNotNull() & F.col('sched_dep').isNotNull() & F.col('sched_arr').isNotNull())
    .withColumn('label', (F.col('arr_delay') >= 15).cast('double'))
    .withColumn('dep_hour', F.floor(F.col('sched_dep') / 100).cast('int') % 24)
    .withColumn('arr_hour', F.floor(F.col('sched_arr') / 100).cast('int') % 24)
    .withColumn('dep_tod', (F.col('dep_hour') * 60 + (F.col('sched_dep') % 100)).cast('double'))
    .withColumn('is_weekend', (F.col('dow') >= 6).cast('double'))
    .withColumn('fold_id', F.pmod(F.xxhash64('month', 'day'), F.lit(10)))
    .withColumn('fold', F.when(F.col('fold_id') < 7, 'train').when(F.col('fold_id') == 7, 'val').otherwise('test'))
    .drop('fold_id', 'arr_delay', 'cancelled', 'diverted', 'sched_dep', 'sched_arr'))

# Плановая загрузка аэропортов: считаем ВСЕ строки расписания, включая отменённые/перенаправленные.
# Это признак, известный из расписания до вылета; не используем фактический статус рейса.
dep_cong = (sel.where(F.col('sched_dep').isNotNull())
    .withColumn('dep_hour', F.floor(F.col('sched_dep') / 100).cast('int') % 24)
    .groupBy('origin', 'month', 'day', 'dep_hour')
    .agg(F.count('*').alias('origin_cong')))
arr_cong = (sel.where(F.col('sched_arr').isNotNull())
    .withColumn('arr_hour', F.floor(F.col('sched_arr') / 100).cast('int') % 24)
    .groupBy('dest', 'month', 'day', 'arr_hour')
    .agg(F.count('*').alias('dest_cong')))
flown = (flown.join(dep_cong, ['origin', 'month', 'day', 'dep_hour'], 'left')
              .join(arr_cong, ['dest', 'month', 'day', 'arr_hour'], 'left'))

if DATA_FRACTION < 1.0:                       # только в local-режиме; расписание аэропортов рассчитано на полном объёме
    flown = flown.sample(False, DATA_FRACTION, SEED)

CLEAN_PATH = SHARED / 'flights_clean.parquet'
with stage('Очистка + загрузка аэропортов + запись parquet'):
    flown.write.mode('overwrite').parquet(spark_path(CLEAN_PATH))

df = spark.read.parquet(spark_path(CLEAN_PATH))
N_USED = df.count()
print(f'после очистки: {N_USED:,} рейсов ({N_USED / N_RAW:.1%} от CSV) · партиций: {df.rdd.getNumPartitions()}')

## 4. EDA на Spark

Агрегаты считаются кластером, на драйвер приходят десятки строк. Те же числа уйдут во фронт (`report.eda`).

In [ ]:
airlines_pd = pd.read_csv(airlines_csv).rename(columns={'IATA_CODE': 'code', 'AIRLINE': 'name'})
AIRLINE_NAMES = dict(zip(airlines_pd.code, airlines_pd.name))


def rate_by(col):
    rows = (df.groupBy(col).agg(F.count('*').alias('n'), F.avg('label').alias('rate')).orderBy(col).collect())
    return [{col: r[col], 'n': int(r['n']), 'rate': float(r['rate'])} for r in rows]


with stage('EDA-агрегаты'):
    eda_hour, eda_month, eda_dow = rate_by('dep_hour'), rate_by('month'), rate_by('dow')
    eda_airline = [{'code': r['airline'], 'name': AIRLINE_NAMES.get(r['airline'], r['airline']), 'n': r['n'], 'rate': r['rate']}
                   for r in rate_by('airline')]
    eda_airline.sort(key=lambda d: -d['rate'])
    fold_rows = df.groupBy('fold').agg(F.count('*').alias('n'), F.avg('label').alias('rate')).collect()

FOLDS = {r['fold']: {'n': int(r['n']), 'prevalence': float(r['rate'])} for r in fold_rows}
PREVALENCE = sum(FOLDS[f]['n'] * FOLDS[f]['prevalence'] for f in FOLDS) / N_USED
print('доля задержек ≥15 мин:', f'{PREVALENCE:.3%}')
for f, v in FOLDS.items():
    print(f'  {f:5s} n={v["n"]:>9,}  prevalence={v["prevalence"]:.3%}')

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
h = pd.DataFrame(eda_hour);  axes[0].bar(h.dep_hour, h.rate * 100, color='#6366f1'); axes[0].set_title('Доля задержек по часу вылета'); axes[0].set_xlabel('час'); axes[0].set_ylabel('%')
a = pd.DataFrame(eda_airline); axes[1].barh(a.code, a.rate * 100, color='#8b5cf6'); axes[1].set_title('По авиакомпаниям'); axes[1].invert_yaxis(); axes[1].set_xlabel('%')
m = pd.DataFrame(eda_month); axes[2].bar(m.month, m.rate * 100, color='#0ea5e9'); axes[2].set_title('По месяцам'); axes[2].set_xlabel('месяц')
plt.tight_layout(); plt.savefig(ART_DIR / 'eda.png', bbox_inches='tight'); plt.show()

## 5. Признаки · расширенная train-only статистика

Для категориальных комбинаций строятся сглаженные частоты задержек только на обучающем сегменте. Это позволяет моделям учитывать риск маршрута и сочетаний перевозчика/времени без прямого использования меток validation/test.

In [ ]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.stat import Summarizer

# Стоимости и целевая функция — прежние, фронтенд использует те же поля.
COST_FP, COST_FN, COST_UNIT = 25.0, 100.0, '$'
COST_POS_WEIGHT = COST_FN / COST_FP  # относительный вес задержанного рейса = 4

train_raw = df.where(F.col('fold') == 'train')
PRIOR = float(train_raw.agg(F.avg('label')).first()[0])

# Сглаженный target encoding. Все таблицы обучаются ТОЛЬКО на train.
TE_SPECS = [
    (['origin'], 'origin_rate', 150.0),
    (['dest'], 'dest_rate', 150.0),
    (['airline'], 'airline_rate', 80.0),
    (['origin', 'dest'], 'route_rate', 200.0),
    (['airline', 'origin', 'dest'], 'airline_route_rate', 500.0),
    (['airline', 'month'], 'airline_month_rate', 150.0),
    (['airline', 'dep_hour'], 'airline_hour_rate', 150.0),
    (['origin', 'dep_hour'], 'origin_hour_rate', 250.0),
    (['dest', 'arr_hour'], 'dest_hour_rate', 250.0),
]

with stage('Расширенные признаки: train-only target encoding + циклическое время'):
    two_pi = 2.0 * math.pi
    feat = (df.fillna({'origin_cong': 1, 'dest_cong': 1})
        .withColumn('hour_sin', F.sin(F.col('dep_hour') * two_pi / 24.0))
        .withColumn('hour_cos', F.cos(F.col('dep_hour') * two_pi / 24.0))
        .withColumn('month_sin', F.sin((F.col('month') - 1) * two_pi / 12.0))
        .withColumn('month_cos', F.cos((F.col('month') - 1) * two_pi / 12.0))
        .withColumn('dow_sin', F.sin((F.col('dow') - 1) * two_pi / 7.0))
        .withColumn('dow_cos', F.cos((F.col('dow') - 1) * two_pi / 7.0))
        .withColumn('log_distance', F.log1p(F.greatest(F.col('distance'), F.lit(0.0))))
        .withColumn('log_origin_cong', F.log1p(F.greatest(F.col('origin_cong'), F.lit(0.0))))
        .withColumn('log_dest_cong', F.log1p(F.greatest(F.col('dest_cong'), F.lit(0.0))))
        .withColumn('scheduled_speed', F.when(F.col('sched_time') > 0, F.col('distance') / F.col('sched_time')).otherwise(0.0))
        .withColumn('is_peak_dep', F.when(F.col('dep_hour').between(7, 10) | F.col('dep_hour').between(16, 19), 1.0).otherwise(0.0))
        .withColumn('is_early_dep', F.when((F.col('dep_hour') < 6) | (F.col('dep_hour') >= 22), 1.0).otherwise(0.0))
    )

    for te_idx, (keys, out_col, smoothing) in enumerate(TE_SPECS):
        # LOO для train: собственная метка строки не участвует в её target-encoded признаке.
        # Для validation/test применяется полная статистика, рассчитанная только по train.
        sum_col, n_col = f'_te_sum_{te_idx}', f'_te_n_{te_idx}'
        te = (train_raw.groupBy(*keys)
              .agg(F.sum('label').alias(sum_col), F.count('*').alias(n_col)))
        feat = feat.join(F.broadcast(te), keys, 'left')
        full_rate = ((F.col(sum_col) + F.lit(smoothing * PRIOR)) /
                     (F.col(n_col) + F.lit(smoothing)))
        loo_rate = ((F.col(sum_col) - F.col('label') + F.lit(smoothing * PRIOR)) /
                    (F.col(n_col) - F.lit(1.0) + F.lit(smoothing)))
        feat = (feat.withColumn(out_col, F.when(F.col('fold') == 'train', loo_rate).otherwise(full_rate))
                    .drop(sum_col, n_col)
                    .fillna({out_col: PRIOR}))

    # Airline OHE для LogReg; StringIndexer/OHE обучаются только на train.
    idx_model = StringIndexer(inputCol='airline', outputCol='airline_idx', handleInvalid='keep').fit(feat.where(F.col('fold') == 'train'))
    feat = idx_model.transform(feat)
    ohe_model = OneHotEncoder(inputCols=['airline_idx'], outputCols=['airline_ohe'], handleInvalid='keep').fit(feat.where(F.col('fold') == 'train'))
    feat = ohe_model.transform(feat)

    TREE_FEATURES = [
        'month', 'dow', 'dep_hour', 'is_weekend',
        'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'dow_sin', 'dow_cos',
        'dep_tod', 'sched_time', 'log_distance', 'scheduled_speed',
        'log_origin_cong', 'log_dest_cong',
        'origin_rate', 'dest_rate', 'airline_rate', 'route_rate',
        'airline_route_rate', 'airline_month_rate', 'airline_hour_rate',
        'origin_hour_rate', 'dest_hour_rate', 'is_peak_dep', 'is_early_dep',
    ]
    LR_NUMERIC = [
        'is_weekend', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'dow_sin', 'dow_cos',
        'log_distance', 'scheduled_speed', 'sched_time', 'log_origin_cong', 'log_dest_cong',
        'origin_rate', 'dest_rate', 'airline_rate', 'route_rate',
        'airline_month_rate', 'airline_hour_rate', 'origin_hour_rate', 'dest_hour_rate',
        'is_peak_dep', 'is_early_dep',
    ]
    AIRLINES = list(idx_model.labelsArray[0])
    LR_NAMES = LR_NUMERIC + [f'airline={a}' for a in AIRLINES]

    feat = VectorAssembler(inputCols=TREE_FEATURES, outputCol='features', handleInvalid='keep').transform(feat)
    feat = VectorAssembler(inputCols=LR_NUMERIC + ['airline_ohe'], outputCol='features_lr', handleInvalid='keep').transform(feat)
    feat = (feat
        .withColumn('cost_weight', F.when(F.col('label') == 1.0, F.lit(COST_POS_WEIGHT)).otherwise(F.lit(1.0)))
        .select('label', 'fold', 'features', 'features_lr', 'cost_weight'))

    train = feat.where(F.col('fold') == 'train').persist(StorageLevel.MEMORY_AND_DISK)
    val   = feat.where(F.col('fold') == 'val').persist(StorageLevel.MEMORY_AND_DISK)
    test  = feat.where(F.col('fold') == 'test').persist(StorageLevel.MEMORY_AND_DISK)
    N_TRAIN, N_VAL, N_TEST = train.count(), val.count(), test.count()

LR_STD = Summarizer.metrics('std').summary(F.col('features_lr')).alias('s')
LR_STD = np.array(train.select(LR_STD).first()['s']['std'].toArray())

print(f'train={N_TRAIN:,}  val={N_VAL:,}  test={N_TEST:,}')
print(f'признаков (деревья): {len(TREE_FEATURES)} | числовых для LogReg: {len(LR_NUMERIC)} + airline OHE: {len(AIRLINES)}')
print('доля задержек train:', f'{PRIOR:.3%}')
print('вес задержки в cost-sensitive варианте:', COST_POS_WEIGHT)

## 6. Модели и денежная функция потерь

Сравниваются прежние семейства моделей, но с расширенными признаками и ограниченным поиском параметров GBT. Cost-sensitive GBT получает вес положительного класса `FN/FP = 4`; это не меняет определение метрик и не заменяет подбор денежного порога.

**Правило выбора:** внутри каждого семейства выбирается конфигурация с минимальной стоимостью на validation. Затем три выбранные модели оцениваются на test; test не используется для выбора модели/параметров/порога.

In [ ]:
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier

TREE_TRAIN_CAP = int(os.environ.get('TREE_TRAIN_CAP', 3_000_000))
frac_tree = min(1.0, TREE_TRAIN_CAP / N_TRAIN)
tree_train = train.sample(False, frac_tree, SEED).persist(StorageLevel.MEMORY_AND_DISK) if frac_tree < 1.0 else train
print(f'деревья учатся на {frac_tree:.0%} train (≈{int(N_TRAIN * frac_tree):,} строк)')


def _cost_per_1000(neg, pos, k, fp_cost=COST_FP, fn_cost=COST_FN):
    tp, fp, fn, tn = hist_counts(neg, pos)
    n = float(tp[0] + fp[0])
    return float((fp[k] * fp_cost + fn[k] * fn_cost) / n * 1000.0) if n else float('nan')


def make_entry(name, family, vh, th, aucs, importance=None, train_s=None,
               mom=None, training_config=None, selected_candidate=None,
               fixed_k=None, is_baseline=False):
    """Формирует старую frontend-схему метрик; порог выбирается только по validation."""
    neg_v, pos_v = vh
    neg_t, pos_t = th
    k = fixed_k if fixed_k is not None else best_threshold_k(neg_v, pos_v, 'cost', COST_FP, COST_FN)
    v_tp, v_fp, v_fn, v_tn = hist_counts(neg_v, pos_v)
    tp, fp, fn, tn = hist_counts(neg_t, pos_t)
    n_t = float(tp[0] + fp[0])
    prev_t_ = float(tp[0] / n_t) if n_t else 0.0
    m = binary_metrics(tp[k], fp[k], fn[k], tn[k])
    m.update(aucs)
    ap = average_precision(neg_t, pos_t)
    brier = brier_from_moments(neg_t, pos_t, mom) if mom is not None else None
    ref = prev_t_ * (1 - prev_t_)
    m.update({'ap': ap, 'brier': brier, 'brier_skill': (1 - brier / ref) if (brier is not None and ref > 0) else None})
    vb = val_best_block(neg_v, pos_v)
    at_f = {}
    for key, v in vb.items():
        kk = int(round(v['threshold'] * NBINS))
        at_f[key] = {'threshold': v['threshold'], **binary_metrics(tp[kk], fp[kk], fn[kk], tn[kk])}
    val_cost = _cost_per_1000(neg_v, pos_v, k)
    # Keep legacy model.type values consumed by the existing report/frontend contract.
    frontend_model_type = {
        'GBT': 'GradientBoosting',
        'LogReg': 'LogisticRegression',
        'RandomForest': 'RandomForest',
        'Baseline': 'Baseline',
    }.get(family, family)
    return {
        'name': name, 'model': frontend_model_type, 'is_baseline': bool(is_baseline), 'is_best': False,
        'threshold': k / NBINS,
        'metrics': m,
        'roc_auc': aucs['roc_auc'], 'pr_auc': aucs['pr_auc'],
        'ap': ap, 'brier': brier,
        'calibration': calibration_block(neg_t, pos_t, mom) if mom is not None else None,
        'val_best': vb, 'at_f_thresholds': at_f,
        'confusion_counts': {'tp': int(tp[k]), 'fp': int(fp[k]), 'fn': int(fn[k]), 'tn': int(tn[k])},
        'cost_per_1000': float((fp[k] * COST_FP + fn[k] * COST_FN) / n_t * 1000) if n_t else None,
        'val_cost_per_1000': val_cost,
        'fixed_05': binary_metrics(tp[NBINS // 2], fp[NBINS // 2], fn[NBINS // 2], tn[NBINS // 2]),
        'sweep': sweep_block(neg_t, pos_t),
        'curves': curves_block(neg_t, pos_t),
        'feature_importance': importance or [],
        'train_seconds': train_s,
        'training_config': training_config or {},
        'selected_candidate': selected_candidate,
    }


def baseline_hist(const_score, like):
    neg, pos = np.zeros(NBINS + 1), np.zeros(NBINS + 1)
    b = 0 if const_score == 0 else NBINS
    neg[b], pos[b] = like[0].sum(), like[1].sum()
    return neg, pos


def baseline_moments(const_score, th):
    neg, pos = th
    mom = {k: np.zeros(NBINS + 1) for k in ('s1_neg', 's1_pos', 's2_neg', 's2_pos')}
    if const_score == 1:
        b = NBINS
        mom['s1_neg'][b] = mom['s2_neg'][b] = neg.sum()
        mom['s1_pos'][b] = mom['s2_pos'][b] = pos.sum()
    return mom


def lr_importance(model, names, std):
    imp = np.abs(model.coefficients.toArray()) * std
    imp = imp / imp.sum() if imp.sum() else imp
    return sorted([{'name': n, 'value': float(v)} for n, v in zip(names, imp)], key=lambda d: -d['value'])


def tree_importance(model, names):
    imp = model.featureImportances.toArray()
    return sorted([{'name': n, 'value': float(v)} for n, v in zip(names, imp)], key=lambda d: -d['value'])


# Варианты GBT различаются глубиной, числом итераций, шагом и регуляризацией.
# Старая конфигурация оставлена как контрольная точка.
candidate_specs = [
    ('GBT_original', 'GBT', tree_train,
     GBTClassifier(featuresCol='features', labelCol='label', maxIter=60, maxDepth=6, stepSize=0.15,
                   subsamplingRate=0.5, maxBins=32, minInstancesPerNode=50, seed=SEED),
     {'maxIter': 60, 'maxDepth': 6, 'stepSize': 0.15, 'subsamplingRate': 0.5, 'maxBins': 32, 'minInstancesPerNode': 50, 'weighted': False}),
    ('GBT_regularized_d4', 'GBT', tree_train,
     GBTClassifier(featuresCol='features', labelCol='label', maxIter=120, maxDepth=4, stepSize=0.07,
                   subsamplingRate=0.8, maxBins=64, minInstancesPerNode=30, seed=SEED),
     {'maxIter': 120, 'maxDepth': 4, 'stepSize': 0.07, 'subsamplingRate': 0.8, 'maxBins': 64, 'minInstancesPerNode': 30, 'weighted': False}),
    ('GBT_cost_sensitive_d4', 'GBT', tree_train,
     GBTClassifier(featuresCol='features', labelCol='label', weightCol='cost_weight', maxIter=120, maxDepth=4,
                   stepSize=0.07, subsamplingRate=0.8, maxBins=64, minInstancesPerNode=30, seed=SEED),
     {'maxIter': 120, 'maxDepth': 4, 'stepSize': 0.07, 'subsamplingRate': 0.8, 'maxBins': 64, 'minInstancesPerNode': 30,
      'weightCol': 'cost_weight', 'positive_weight': COST_POS_WEIGHT}),
    ('GBT_interactions_d5', 'GBT', tree_train,
     GBTClassifier(featuresCol='features', labelCol='label', maxIter=90, maxDepth=5, stepSize=0.06,
                   subsamplingRate=0.75, maxBins=64, minInstancesPerNode=40, seed=SEED),
     {'maxIter': 90, 'maxDepth': 5, 'stepSize': 0.06, 'subsamplingRate': 0.75, 'maxBins': 64, 'minInstancesPerNode': 40, 'weighted': False}),
    ('RandomForest_tuned', 'RandomForest', tree_train,
     RandomForestClassifier(featuresCol='features', labelCol='label', numTrees=80, maxDepth=14, maxBins=64,
                            subsamplingRate=0.5, minInstancesPerNode=20, featureSubsetStrategy='sqrt', seed=SEED),
     {'numTrees': 80, 'maxDepth': 14, 'maxBins': 64, 'subsamplingRate': 0.5, 'minInstancesPerNode': 20, 'featureSubsetStrategy': 'sqrt'}),
    ('LogReg_regularized', 'LogReg', train,
     LogisticRegression(featuresCol='features_lr', labelCol='label', maxIter=100, regParam=0.01,
                        elasticNetParam=0.05, standardization=True),
     {'maxIter': 100, 'regParam': 0.01, 'elasticNetParam': 0.05, 'standardization': True}),
]

candidate_results = []
for candidate_name, family, tr, estimator, config in candidate_specs:
    with stage(f'обучение · {candidate_name}'):
        model = estimator.fit(tr)
    train_s = TIMINGS[f'обучение · {candidate_name}']
    with stage(f'валидация · {candidate_name}'):
        val_s = score_sdf(model, val).persist()
        vh = score_histogram(val_s)
        val_s.unpersist()
    k_val = best_threshold_k(vh[0], vh[1], 'cost', COST_FP, COST_FN)
    val_cost = _cost_per_1000(vh[0], vh[1], k_val)
    val_ap = average_precision(vh[0], vh[1])
    candidate_results.append({
        'candidate': candidate_name, 'family': family, 'model': model, 'val_hist': vh,
        'val_threshold': k_val / NBINS, 'val_cost_per_1000': val_cost,
        'val_ap': val_ap, 'train_seconds': train_s, 'config': config,
    })
    print(f'  {candidate_name}: val threshold={k_val/NBINS:.3f} · val cost={val_cost:,.0f}{COST_UNIT}/1000 · AP(val)={val_ap:.4f}')

# Выбираем конфигурацию отдельно в каждом семействе ТОЛЬКО по validation.
family_winners = {}
for family in ('GBT', 'RandomForest', 'LogReg'):
    options = [x for x in candidate_results if x['family'] == family]
    family_winners[family] = min(options, key=lambda x: (x['val_cost_per_1000'], -x['val_ap']))

experiment_tbl = pd.DataFrame([{
    'family': x['family'], 'candidate': x['candidate'], 'val_threshold': round(x['val_threshold'], 3),
    f'val_cost_per_1000_{COST_UNIT}': round(x['val_cost_per_1000']), 'val_AP': round(x['val_ap'], 4),
    'train_seconds': round(x['train_seconds'], 1), 'config': json.dumps(x['config'], ensure_ascii=False),
} for x in candidate_results]).sort_values(f'val_cost_per_1000_{COST_UNIT}')
display(experiment_tbl)
experiment_tbl.to_csv(ART_DIR / 'training_experiments.csv', index=False)

# Финальная оценка: test считаем только для победителей семейств.
entries, fitted, hists = [], {}, {}
for family, chosen in family_winners.items():
    name = family  # Имена семейств LogReg / RandomForest / GBT остаются совместимы с фронтендом.
    model = chosen['model']
    with stage(f'тест · {name}'):
        test_s = score_sdf(model, test).persist()
        neg_t, pos_t, mom_t = score_histogram(test_s, moments=True)
        th = (neg_t, pos_t)
        aucs = spark_auc(test_s)
        test_s.unpersist()
    imp = lr_importance(model, LR_NAMES, LR_STD) if family == 'LogReg' else tree_importance(model, TREE_FEATURES)
    entry = make_entry(name, family, chosen['val_hist'], th, aucs, imp, chosen['train_seconds'], mom_t,
                       training_config=chosen['config'], selected_candidate=chosen['candidate'])
    entries.append(entry)
    fitted[name], hists[name] = model, (chosen['val_hist'], th)
    print(f'  {name}: candidate={chosen["candidate"]} · val cost={entry["val_cost_per_1000"]:,.0f}{COST_UNIT}/1000'
          f' · test cost={entry["cost_per_1000"]:,.0f}{COST_UNIT}/1000 · AP(test)={entry["ap"]:.4f} · Brier={entry["brier"]:.4f}')

# Baselines строятся на метках тех же validation/test сегментов, что и выбранные модели.
ref_family = 'GBT' if 'GBT' in hists else next(iter(hists))
ref_vh, ref_th = hists[ref_family]
prev_t = float(ref_th[1].sum() / (ref_th[0].sum() + ref_th[1].sum()))
for bname, const in (('Baseline · всегда вовремя', 0), ('Baseline · всегда задержка', 1)):
    vh = baseline_hist(const, ref_vh)
    th = baseline_hist(const, ref_th)
    entries.append(make_entry(bname, 'Baseline', vh, th, {'roc_auc': 0.5, 'pr_auc': prev_t},
                              fixed_k=NBINS // 2, is_baseline=True,
                              mom=baseline_moments(const, th), training_config={'constant_score': const}))

models_only = [e for e in entries if not e['is_baseline']]
# Важное отличие от старого файла: победитель выбирается по validation cost, не по test cost.
best = min(models_only, key=lambda e: (e['val_cost_per_1000'], -e['ap']))
best['is_best'] = True
print('\nЛучшая по validation cost:', best['name'], f'{best["val_cost_per_1000"]:,.0f}{COST_UNIT}/1000')
print('Её независимая тестовая стоимость:', f'{best["cost_per_1000"]:,.0f}{COST_UNIT}/1000')

## 7. Сводка

Читаем одновременно **цену**, **AP**, **Brier Score** и **MCC**: модель может выигрывать по ROC-AUC, а по деньгам — нет, если стоимости ошибок несимметричны.
AP (ступенчатая PR-площадь) заменяет ROC-кривую: она считается только по положительному классу и не раздувается числом верно отпущенных рейсов.
Brier Score оценивает сами вероятности — отдельно от порога (ниже — лучше, ориентир `prev·(1−prev)`).

In [ ]:
tbl = pd.DataFrame([{
    'model': e['name'], 'candidate': e.get('selected_candidate') or e['name'],
    'val cost/1000': round(e.get('val_cost_per_1000', float('nan'))),
    'thr': round(e['threshold'], 3), f'cost/1000 {COST_UNIT}': round(e['cost_per_1000']),
    'FP': e['confusion_counts']['fp'], 'FN': e['confusion_counts']['fn'],
    'Sensitivity': round(e['metrics']['sensitivity'], 4), 'Specificity': round(e['metrics']['specificity'], 4),
    'Precision': round(e['metrics']['precision_pos'], 4), 'NPV': round(e['metrics']['npv'], 4),
    'MCC': round(e['metrics']['mcc'], 4), 'AP': round(e['ap'], 4), 'Brier': round(e['brier'], 4),
    'F0.5': round(e['metrics']['f0.5_pos'], 4), 'F2.0': round(e['metrics']['f2_pos'], 4),
    'ROC-AUC': round(e['roc_auc'], 4), 'Train s': round(e['train_seconds'], 1) if e['train_seconds'] is not None else None,
} for e in entries]).set_index('model')
display(tbl)
tbl.to_csv(ART_DIR / 'metrics_all_models.csv')


def pr_steps(e):
    """Ступенчатая PR-кривая (precision справа от шага — как в формуле AP)."""
    return e['curves']['pr']['recall'], e['curves']['pr']['precision']

# ---- 1. цена ошибок + PR-кривые ----
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for e in entries:
    sw = e['sweep']
    n_t = sw['tp'][0] + sw['fp'][0]
    cost = [(f * COST_FP + n * COST_FN) / n_t * 1000 for f, n in zip(sw['fp'], sw['fn'])]
    axes[0].plot(sw['thresholds'], cost, '--' if e['is_baseline'] else '-', lw=2,
                 label=f'{e["name"]} · val={e.get("val_cost_per_1000", float("nan")):,.0f}')
    if not e['is_baseline']:
        r, p = pr_steps(e)
        axes[1].step(r, p, where='pre', lw=2, label=f'{e["name"]} · AP={e["ap"]:.3f}')
axes[0].set_xlabel('порог'); axes[0].set_ylabel(f'потери на 1000 рейсов, {COST_UNIT}')
axes[0].set_title(f'Цена ошибок (FP={COST_FP:.0f}{COST_UNIT}, FN={COST_FN:.0f}{COST_UNIT})'); axes[0].legend(fontsize=8)
axes[1].axhline(prev_t, color='k', ls='--', alpha=.5, label=f'test prevalence={prev_t:.3f}')
axes[1].set_xlabel('recall'); axes[1].set_ylabel('precision'); axes[1].set_title('Precision–Recall (test)'); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig(ART_DIR / 'cost_and_pr.png', bbox_inches='tight'); plt.show()

# ---- 2. AP, Brier Score, калибровка ----
trained = [e for e in entries if not e['is_baseline']]
names = [e['name'].replace('Baseline · ', 'base · ') for e in entries]
colors = ['#94a3b8' if e['is_baseline'] else '#6366f1' for e in entries]
best_e = next(e for e in entries if e['is_best'])
brier_ref = prev_t * (1 - prev_t)

fig, axes = plt.subplots(1, 3, figsize=(18, 4.8))
ax = axes[0]
bars = ax.bar(names, [e['ap'] for e in entries], color=colors)
ax.axhline(prev_t, color='k', ls='--', alpha=.5, label=f'без модели = test prevalence {prev_t:.3f}')
for b_, e in zip(bars, entries):
    ax.text(b_.get_x() + b_.get_width() / 2, b_.get_height(), f'{e["ap"]:.3f}', ha='center', va='bottom', fontsize=9)
ax.set_title('Average Precision (выше — лучше)'); ax.set_ylabel('AP'); ax.legend(fontsize=8); ax.tick_params(axis='x', rotation=20)

ax = axes[1]
bars = ax.bar(names, [e['brier'] for e in entries], color=colors)
ax.axhline(brier_ref, color='k', ls='--', alpha=.5, label=f'no-skill prev·(1−prev) = {brier_ref:.4f}')
for b_, e in zip(bars, entries):
    ax.text(b_.get_x() + b_.get_width() / 2, b_.get_height(), f'{e["brier"]:.4f}', ha='center', va='bottom', fontsize=9)
ax.set_title('Brier Score (ниже — лучше)'); ax.set_ylabel('Brier'); ax.legend(fontsize=8); ax.tick_params(axis='x', rotation=20)

ax = axes[2]
ax.plot([0, 1], [0, 1], 'k--', alpha=.5, label='идеальная калибровка')
for e in trained:
    c = e['calibration']
    ax.plot(c['mean_pred'], c['obs_rate'], 'o-', lw=2.5 if e['is_best'] else 1.5, label=f'{e["name"]} · ECE={c["ece"]:.3f}')
ax.set_xlabel('предсказанная вероятность'); ax.set_ylabel('фактическая доля задержек')
ax.set_title('Калибровка (10 интервалов, test)'); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(ART_DIR / 'ap_brier.png', bbox_inches='tight'); plt.show()

### Порог по F0.5 и F2 — для положительного и отрицательного сценария

* **Положительный сценарий** — целевой класс «задержка»: precision = TP/(TP+FP), recall = Sensitivity.
* **Отрицательный сценарий** — целевой класс «вовремя»: precision = **NPV** = TN/(TN+FN), recall = **Specificity**.

F0.5 делает упор на precision (меньше ложных тревог), F2 — на recall (меньше пропусков). Пороги подбираются на **валидации** (`val_best`),
метрики при них считаются на тесте (`at_f_thresholds`). Во фронте у каждого из четырёх оптимумов есть кнопка «выбрать лучший».

Ниже — confusion matrix **в стандартном виде**: строки — фактический класс, столбцы — предсказанный, положительный класс первым
(`[[TP, FN], [FP, TN]]`). Внимание: у `sklearn.metrics.confusion_matrix` порядок обратный — `[[TN, FP], [FN, TP]]`.

In [ ]:
# ---- пороги по F0.5 / F2 для лучшей модели ----
fc_test = f_curves(*hists[best_e['name']][1])
ks = np.arange(NBINS + 1) / NBINS
fig, axes = plt.subplots(1, 2, figsize=(15, 4.6))
for ax, side, title in ((axes[0], 'pos', 'Положительный сценарий: «задержка»'), (axes[1], 'neg', 'Отрицательный сценарий: «вовремя»')):
    for beta, col in (('f0.5', '#0ea5e9'), ('f2', '#f59e0b')):
        key = f'{beta}_{side}'
        ax.plot(ks, fc_test[key], color=col, lw=2, label=beta.upper())
        kv = int(round(best_e['val_best'][key]['threshold'] * NBINS))          # порог, выбранный на валидации
        kt = int(np.argmax(fc_test[key]))                                      # максимум на тесте (для сравнения)
        ax.scatter([ks[kv]], [fc_test[key][kv]], color=col, s=70, zorder=5, label=f'{beta} · порог val = {ks[kv]:.3f}')
        ax.scatter([ks[kt]], [fc_test[key][kt]], color=col, s=70, zorder=5, marker='x', label=f'{beta} · максимум на тесте = {ks[kt]:.3f}')
    ax.axvline(best_e['threshold'], color='#6366f1', ls='--', alpha=.7, label=f'порог по цене = {best_e["threshold"]:.3f}')
    ax.set_title(f'{title} — {best_e["name"]}'); ax.set_xlabel('порог'); ax.set_ylabel('F-мера'); ax.set_ylim(0, 1.02); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(ART_DIR / 'fbeta_threshold.png', bbox_inches='tight'); plt.show()

rows_f = []
for key, v in best_e['at_f_thresholds'].items():
    rows_f.append({'оптимум': key, 'порог (val)': round(v['threshold'], 3), 'Sensitivity': round(v['sensitivity'], 4),
                   'Specificity': round(v['specificity'], 4), 'Precision': round(v['precision_pos'], 4), 'NPV': round(v['npv'], 4),
                   'MCC': round(v['mcc'], 4), 'F0.5': round(v['f0.5_pos'], 4), 'F2.0': round(v['f2_pos'], 4)})
display(pd.DataFrame(rows_f).set_index('оптимум'))

# ---- confusion matrix: стандартный вид [[TP, FN], [FP, TN]] ----
cc_ = best_e['confusion_counts']
cm_std = np.array([[cc_['tp'], cc_['fn']], [cc_['fp'], cc_['tn']]])
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
for ax, data, fmt, ttl in ((axes[0], cm_std, ',d', 'Количество'), (axes[1], cm_std / cm_std.sum(axis=1, keepdims=True), '.1%', 'Доля от строки')):
    sns.heatmap(data, annot=True, fmt=fmt, cmap='Blues', cbar=False, ax=ax, annot_kws={'size': 14},
                xticklabels=['Positive (задержка)', 'Negative (вовремя)'], yticklabels=['Positive (задержка)', 'Negative (вовремя)'])
    ax.set_xlabel('Predicted (предсказано)'); ax.set_ylabel('Actual (факт)'); ax.set_title(f'{ttl} · {best_e["name"]}, порог {best_e["threshold"]:.3f}')
plt.tight_layout(); plt.savefig(ART_DIR / 'confusion_matrix.png', bbox_inches='tight'); plt.show()
print(f'Sensitivity={cc_["tp"] / (cc_["tp"] + cc_["fn"]):.4f}  Specificity={cc_["tn"] / (cc_["tn"] + cc_["fp"]):.4f}  '
      f'Precision={cc_["tp"] / (cc_["tp"] + cc_["fp"]):.4f}  NPV={cc_["tn"] / (cc_["tn"] + cc_["fn"]):.4f}')

## 8. Сохранение: `report.json`, `index.json`, модель

Фронт читает `artifacts/classification/flight_delays/report.json`. NaN/Inf заменяются на `null` (иначе `JSON.parse` в браузере падает).
Текст вывода собирается из реально посчитанных чисел — ничего не вписано вручную.

In [ ]:
def pct_(x, d=1):
    return f'{x * 100:.{d}f}%'


best_baseline = min((e for e in entries if e['is_baseline']), key=lambda e: (e['val_cost_per_1000'], -e['ap']))
saving = 1 - best['cost_per_1000'] / best_baseline['cost_per_1000']
cc = best['confusion_counts']
best_auc_name = max(models_only, key=lambda e: e['roc_auc'])['name']
best_ap_name = max(models_only, key=lambda e: e['ap'])['name']
best_brier_name = min(models_only, key=lambda e: e['brier'])['name']
test_prevalence = float(FOLDS['test']['prevalence'])
brier_ref_ = test_prevalence * (1 - test_prevalence)
theory_thr = COST_FP / (COST_FP + COST_FN)

SPARK_BLOCK = {**SPARK_INFO, 'n_rows_raw': int(N_RAW), 'n_rows_used': int(N_USED), 'timings': TIMINGS,
               'tree_train_rows': int(N_TRAIN * frac_tree), 'ui': {'master': ':8085', 'history': ':18080'}}

narrative = {
    'context': (f'BTS 2015: {N_RAW:,} рейсов, после очистки {N_USED:,}. Задержкой считаем прибытие позже плана на ≥15 минут '
                f'({pct_(PREVALENCE)} рейсов). Признаки — расписание, перевозчик, маршрут, плановая загрузка аэропортов по расписанию и сглаженная статистика '
                'по комбинациям перевозчика/маршрута/часа. Target encoding рассчитывается только на train. '
                'Разбиение по календарным дням исключает пересечение одного дня между train/val/test.'),
    'hero_metric': f'Потери на 1000 рейсов, {COST_UNIT}',
    'why_hero': (f'FP стоит {COST_FP:.0f}{COST_UNIT}, FN — {COST_FN:.0f}{COST_UNIT}: ошибка пропуска задержки в четыре раза дороже ложной тревоги. '
                'Порог выбирается по стоимости на validation, test остаётся финальной проверкой.'),
    'why_not_accuracy': (f'Baseline «всегда вовремя» имеет accuracy {pct_(next(e for e in entries if e["name"].endswith("вовремя"))["metrics"]["accuracy"])} '
                         'и при этом не ловит ни одной задержки. Accuracy не отражает цену пропусков.'),
    'main_conclusion': (f'Выбранная по validation модель — {best["name"]} ({best.get("selected_candidate") or best["name"]}): '
                        f'{best["cost_per_1000"]:,.0f}{COST_UNIT}/1000 на test против {best_baseline["cost_per_1000"]:,.0f}{COST_UNIT} '
                        f'у лучшего baseline ({pct_(saving)} экономии). На validation стоимость составляла '
                        f'{best["val_cost_per_1000"]:,.0f}{COST_UNIT}/1000; порог {best["threshold"]:.3f}. '
                        f'На test поймано {cc["tp"]:,} задержек, пропущено {cc["fn"]:,}, ложных тревог {cc["fp"]:,}. '
                        'Выбор модели и порога не использовал test.'),
    'why_ap_brier': (f'AP оценивает ранжирование задержек; случайный уровень на test равен test prevalence = {pct_(test_prevalence)}, '
                     f'лучшая модель ({best_ap_name}) — {max(e["ap"] for e in models_only):.3f}. '
                     f'Brier оценивает качество вероятностей: no-skill prev·(1−prev) на test = {brier_ref_:.4f}, '
                     f'лучшая ({best_brier_name}) — {min(e["brier"] for e in models_only):.4f}. '
                     'Пороги F0.5/F2 отдельно для обоих классов подобраны на validation.'),
    'rules': {
        'cost_first': 'Порог выбирается по цене ошибок на validation, а не по умолчанию 0.5',
        'no_leakage': 'Target encoding и кодировщики категорий обучаются только на train; для train используется leave-one-out. Загрузка аэропортов считается по опубликованному расписанию, не по факту выполненных рейсов.',
        'test_holdout': 'Выбор конфигурации выполняется на validation; test используется только для итоговой оценки выбранных моделей',
        'baselines': 'Любая модель сравнивается с «всегда вовремя» и «всегда задержка» на тех же метках test',
        'ap_not_roc': 'На дисбалансе ранжирование оцениваем по AP; вероятности — по Brier Score',
    },
}

DIAG = {
    'is_balanced': False,
    'prevalence_positive': float(PREVALENCE),
    'test_prevalence_positive': float(test_prevalence),
    'best_cost_per_1000': float(best['cost_per_1000']),
    'best_validation_cost_per_1000': float(best['val_cost_per_1000']),
    'baseline_cost_per_1000': float(best_baseline['cost_per_1000']),
    'saving_vs_baseline': float(saving),
    'best_pr_auc': float(best['pr_auc']),
    'best_roc_auc': float(best['roc_auc']),
    'best_mcc': float(best['metrics']['mcc']),
    'best_ap': float(best['ap']),
    'best_brier': float(best['brier']),
    'best_brier_skill': None if best['metrics']['brier_skill'] is None else float(best['metrics']['brier_skill']),
    'tp': cc['tp'], 'fp': cc['fp'], 'fn': cc['fn'], 'tn': cc['tn'],
    'cost_optimal_threshold_theory': theory_thr,
    'val_threshold': float(best['threshold']),
    'selected_candidate': best.get('selected_candidate'),
}

report = {
    'meta': {
        'task': TASK,
        'title': 'Flight Delays · цена ошибки',
        'subtitle': 'Порог выбирается по деньгам: FN дороже FP — оптимум смещается от 0.5',
        'dataset': 'usdot/flight-delays (Kaggle · BTS 2015)',
        'target': 'ARRIVAL_DELAY ≥ 15 мин',
        'classes': ['on_time', 'delayed'], 'positive_class': 'delayed',
        'features': TREE_FEATURES, 'seed': SEED, 'generated_at': datetime.now().isoformat(),
        'n_samples_raw': int(N_RAW), 'n_samples': int(N_USED), 'n_features': len(TREE_FEATURES),
        'n_train': int(N_TRAIN), 'n_val': int(N_VAL), 'n_test': int(N_TEST),
        'target_stats': {
            'prevalence': float(PREVALENCE),
            'prevalence_by_fold': {f: v['prevalence'] for f, v in FOLDS.items()},
            'imbalance_ratio': float((1 - PREVALENCE) / PREVALENCE),
        },
        'costs': {'fp': COST_FP, 'fn': COST_FN, 'unit': COST_UNIT},
        'split_note': 'Разбиение целыми календарными днями по hash(month, day) % 10 → 0–6 train, 7 val, 8–9 test; это day-grouped split, не хронологический holdout.',
        'training_note': 'Модель и порог выбираются по validation cost; test не участвует в отборе',
        'training_candidates': [
            {'candidate': x['candidate'], 'family': x['family'], 'val_threshold': float(x['val_threshold']),
             'val_cost_per_1000': float(x['val_cost_per_1000']), 'val_ap': float(x['val_ap']),
             'train_seconds': float(x['train_seconds']), 'config': x['config']}
            for x in candidate_results
        ],
        'selected_training_configs': {family: {
            'candidate': chosen['candidate'], 'config': chosen['config'],
            'validation_cost_per_1000': float(chosen['val_cost_per_1000']),
        } for family, chosen in family_winners.items()},
        'spark': SPARK_BLOCK,
        'narrative': narrative,
    },
    'headline': {
        'best_model': best['name'], 'best_metric': 'cost_per_1000', 'best_metric_value': float(best['cost_per_1000']),
        'verdict': (f'{best["name"]} · {best["cost_per_1000"]:,.0f}{COST_UNIT}/1000 · экономия {pct_(saving)} · '
                    f'sens={pct_(best["metrics"]["sensitivity"])} · spec={pct_(best["metrics"]["specificity"])} · '
                    f'prec={pct_(best["metrics"]["precision_pos"])} · AP={best["ap"]:.3f} · Brier={best["brier"]:.4f}'),
        'diagnostic_verdict': DIAG,
    },
    # Основные поля каждой модели не изменены; добавлены validation_cost_per_1000/config.
    'models': sorted(entries, key=lambda e: (e['is_baseline'], e.get('val_cost_per_1000', float('inf')))),
    'eda': {'by_hour': eda_hour, 'by_month': eda_month, 'by_dow': eda_dow, 'by_airline': eda_airline},
    'charts': {k.replace('.png', ''): k for k in ('eda.png', 'cost_and_pr.png', 'ap_brier.png', 'fbeta_threshold.png', 'confusion_matrix.png') if (ART_DIR / k).exists()},
}

# Жёсткая проверка совместимости с текущим components/flights/FlightDelaysPage.tsx.
# Если контракт изменится случайно, ноутбук остановится ДО перезаписи report.json.
def validate_frontend_contract(payload):
    meta = payload.get('meta', {})
    required_meta = {
        'title', 'subtitle', 'n_samples', 'n_train', 'n_val', 'n_test',
        'n_features', 'features', 'target_stats', 'costs', 'spark', 'narrative', 'split_note',
    }
    missing_meta = sorted(required_meta - set(meta))
    if missing_meta:
        raise AssertionError(f'Frontend contract: missing report.meta keys: {missing_meta}')

    for key in ('prevalence',):
        if key not in meta['target_stats']:
            raise AssertionError(f'Frontend contract: missing meta.target_stats.{key}')
    for key in ('fp', 'fn', 'unit'):
        if key not in meta['costs']:
            raise AssertionError(f'Frontend contract: missing meta.costs.{key}')
    for key in ('timings', 'mode', 'spark_version', 'n_rows_raw', 'n_rows_used',
                'cores', 'executor_memory', 'default_parallelism', 'shuffle_partitions',
                'data_fraction', 'shared_dir', 'ui'):
        if key not in meta['spark']:
            raise AssertionError(f'Frontend contract: missing meta.spark.{key}')
    for key in ('context', 'why_hero', 'why_not_accuracy', 'why_ap_brier', 'main_conclusion', 'rules'):
        if key not in meta['narrative']:
            raise AssertionError(f'Frontend contract: missing meta.narrative.{key}')

    expected_model_names = {'GBT', 'LogReg', 'RandomForest',
                            'Baseline · всегда вовремя', 'Baseline · всегда задержка'}
    models = payload.get('models', [])
    actual_model_names = {m.get('name') for m in models}
    if not expected_model_names.issubset(actual_model_names):
        raise AssertionError(
            f'Frontend contract: missing model names: {sorted(expected_model_names - actual_model_names)}'
        )
    if sum(bool(m.get('is_best')) for m in models) != 1:
        raise AssertionError('Frontend contract: exactly one model must have is_best=true')

    required_model_keys = {
        'name', 'model', 'is_baseline', 'is_best', 'threshold', 'metrics',
        'roc_auc', 'pr_auc', 'ap', 'brier', 'calibration', 'val_best',
        'at_f_thresholds', 'confusion_counts', 'cost_per_1000', 'fixed_05',
        'sweep', 'curves', 'feature_importance', 'train_seconds',
    }
    for model in models:
        missing = sorted(required_model_keys - set(model))
        if missing:
            raise AssertionError(f"Frontend contract: {model.get('name')} missing keys: {missing}")
        required_metrics = {
            'accuracy', 'balanced_accuracy', 'f1_macro', 'f1_weighted', 'mcc', 'kappa',
            'sensitivity', 'specificity', 'precision_pos', 'npv', 'f0.5_pos', 'f2_pos',
            'f0.5_neg', 'f2_neg', 'roc_auc', 'pr_auc', 'ap', 'brier', 'brier_skill',
        }
        missing_metrics = sorted(required_metrics - set(model['metrics']))
        if missing_metrics:
            raise AssertionError(f"Frontend contract: {model['name']} missing metrics: {missing_metrics}")
        for key in ('tp', 'fp', 'fn', 'tn'):
            if key not in model['confusion_counts']:
                raise AssertionError(f"Frontend contract: {model['name']} missing confusion_counts.{key}")
        sweep = model['sweep']
        for key in ('thresholds', 'tp', 'fp', 'fn', 'tn'):
            if key not in sweep:
                raise AssertionError(f"Frontend contract: {model['name']} missing sweep.{key}")
        sweep_lengths = {len(sweep[k]) for k in ('thresholds', 'tp', 'fp', 'fn', 'tn')}
        if len(sweep_lengths) != 1 or not sweep.get('thresholds'):
            raise AssertionError(f"Frontend contract: {model['name']} has inconsistent sweep arrays")
        pr = model['curves'].get('pr', {})
        if not pr.get('recall') or len(pr.get('recall', [])) != len(pr.get('precision', [])):
            raise AssertionError(f"Frontend contract: {model['name']} has invalid curves.pr")
        calibration = model.get('calibration')
        if not calibration or not all(k in calibration for k in ('mean_pred', 'obs_rate', 'n', 'ece')):
            raise AssertionError(f"Frontend contract: {model['name']} has invalid calibration")
        if not (len(calibration['mean_pred']) == len(calibration['obs_rate']) == len(calibration['n'])):
            raise AssertionError(f"Frontend contract: {model['name']} has inconsistent calibration arrays")
        if model['model'] not in ('GradientBoosting', 'LogisticRegression', 'RandomForest', 'Baseline'):
            raise AssertionError(f"Frontend contract: unexpected legacy model type {model['model']!r}")
        if not isinstance(model.get('feature_importance'), list):
            raise AssertionError(f"Frontend contract: {model['name']} feature_importance must be a list")
        for item in model['feature_importance']:
            if not {'name', 'value'}.issubset(item):
                raise AssertionError(f"Frontend contract: {model['name']} feature importance item has invalid shape")

    eda = payload.get('eda', {})
    for key in ('by_hour', 'by_month', 'by_airline'):
        if key not in eda or not isinstance(eda[key], list):
            raise AssertionError(f'Frontend contract: missing/invalid report.eda.{key}')
    for row in eda['by_hour']:
        if not {'dep_hour', 'n', 'rate'}.issubset(row):
            raise AssertionError('Frontend contract: invalid EDA hour row')
    for row in eda['by_month']:
        if not {'month', 'n', 'rate'}.issubset(row):
            raise AssertionError('Frontend contract: invalid EDA month row')
    for row in eda['by_airline']:
        if not {'name', 'n', 'rate'}.issubset(row):
            raise AssertionError('Frontend contract: invalid EDA airline row')

    if not payload.get('headline', {}).get('best_model'):
        raise AssertionError('Frontend contract: missing headline.best_model')
    return True

validate_frontend_contract(report)
print('✓ report.json: frontend contract verified (no frontend changes required)')

# --- модель и метаданные ---
slug = best['name'].lower().replace(' ', '_')
spark_model_path = MODEL_DIR / f'flight_delays_{slug}_spark'
try:
    fitted[best['name']].write().overwrite().save(spark_model_path.as_uri())
    saved_model = str(spark_model_path.relative_to(PROJECT))
except Exception as e:
    print('⚠ модель Spark не сохранена:', type(e).__name__, str(e)[:200])
    saved_model = None

meta_path = MODEL_DIR / f'flight_delays_{slug}_{datetime.now():%Y%m%d_%H%M}.json'
dump_json({'task': TASK, 'model': best['name'], 'selected_candidate': best.get('selected_candidate'),
           'features': TREE_FEATURES, 'threshold': best['threshold'], 'training_config': best.get('training_config'),
           'costs': report['meta']['costs'], 'metrics': best['metrics'], 'saved_at': datetime.now().isoformat(),
           'spark_model': saved_model}, meta_path)

report['artifacts'] = {'model_spark': saved_model, 'model_meta': str(meta_path.relative_to(PROJECT)),
                       'metrics_csv': str((ART_DIR / 'metrics_all_models.csv').relative_to(PROJECT)),
                       'training_experiments_csv': str((ART_DIR / 'training_experiments.csv').relative_to(PROJECT))}
report_path = ART_DIR / 'report.json'
dump_json(report, report_path)


def upsert_index(entry):
    p = PROJECT / 'artifacts' / 'index.json'
    idx = json.loads(p.read_text()) if p.exists() else {'generated_at': None, 'tasks': []}
    idx['tasks'] = sorted([t for t in idx['tasks'] if t['id'] != entry['id']] + [entry], key=lambda t: t['id'])
    idx['generated_at'] = datetime.now().isoformat()
    dump_json(idx, p)


upsert_index({'id': TASK, 'title': report['meta']['title'], 'best_model': best['name'],
              'hero_metric': 'cost_per_1000', 'hero_value': float(best['cost_per_1000']), 'unit': COST_UNIT,
              'report': str(report_path.relative_to(PROJECT)), 'charts_dir': str(ART_DIR.relative_to(PROJECT))})

print('report →', report_path, f'({report_path.stat().st_size / 1024:.0f} КБ)')
print(report['headline']['verdict'])
print('selected candidate:', best.get('selected_candidate'))
print('время этапов:', json.dumps(TIMINGS, ensure_ascii=False))

## 9. Итог

* Данные, целевая метка, стоимость FP/FN, названия ключевых метрик и формат `report.json` сохранены.
* Изменено обучение: расширенные train-only target encodings, циклические признаки времени, регуляризация GBT и cost-sensitive GBT с весами 4:1.
* Конфигурации выбираются по стоимости на validation. Test используется только для оценки трёх финальных семей; выбор победителя по test устранён.
* В `models[]` сохранены прежние ключи для фронтенда: `metrics`, `ap`, `brier`, `threshold`, `confusion_counts`, `sweep`, `curves`, `feature_importance`, `train_seconds` и др.
* Для аудита дополнительно создаётся `training_experiments.csv` и в `meta` записываются выбранные параметры.
* Не путать prevalence `(TP + FN) / N` с долей рейсов, помеченных моделью как задержанные `(TP + FP) / N`: вторая зависит от порога и меняется при выборе GBT/LogReg/RandomForest или baseline.
* Новые значения Sensitivity/Precision/AP/Brier и потерь появятся только после полного запуска этого ноутбука на Spark. Локальный fallback с неполной выборкой нельзя сопоставлять с полным предыдущим прогоном.

**Артефакты:** `artifacts/classification/flight_delays/report.json`, `metrics_all_models.csv`, `training_experiments.csv`, графики и `models/classification/flight_delays/…`.

Остановить Spark и освободить ресурсы: `spark.stop()`.